# CNN MNIST avec Visualisation des Embeddings

## Objectifs :
- Construire un réseau de neurones convolutif (CNN) pour classifier les chiffres MNIST
- Extraire les embeddings (représentations) de la dernière couche cachée
- Visualiser les embeddings en 2D avec t-SNE et UMAP

Auteur : [Papa Séga WADE](https://papasegawade.com)

## 1. Import des bibliothèques

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import Dense, Conv2D, MaxPooling2D, Flatten, Dropout
from tensorflow.keras.utils import to_categorical

# Pour la visualisation des embeddings
from sklearn.manifold import TSNE
try:
    import umap
    UMAP_AVAILABLE = True
except ImportError:
    UMAP_AVAILABLE = False
    print("UMAP n'est pas installé. Seulement t-SNE sera utilisé.")

print(f"TensorFlow version: {tf.__version__}")

## 2. Chargement et Préparation des Données MNIST

In [ ]:
# Chargement des données
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print(f"Forme des données d'entraînement: {X_train.shape}")
print(f"Forme des données de test: {X_test.shape}")

# Affichage de quelques exemples
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f'Label: {y_train[i]}')
    ax.axis('off')
plt.suptitle('Exemples d\'images MNIST', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Reshape pour CNN (ajout du canal de couleur)
X_train = X_train.reshape(-1, 28, 28, 1).astype('float32')
X_test = X_test.reshape(-1, 28, 28, 1).astype('float32')

# Normalisation
X_train /= 255.0
X_test /= 255.0

# One-hot encoding des labels
num_classes = 10
y_train_cat = to_categorical(y_train, num_classes)
y_test_cat = to_categorical(y_test, num_classes)

print(f"Forme finale X_train: {X_train.shape}")
print(f"Forme finale y_train: {y_train_cat.shape}")

## 3. Construction du Modèle CNN

In [ ]:
def create_cnn_model():
    """
    Crée un CNN pour la classification MNIST
    Architecture:
    - Conv2D (32 filters) -> MaxPooling
    - Conv2D (64 filters) -> MaxPooling
    - Conv2D (128 filters) -> MaxPooling
    - Flatten
    - Dense (128) -> Embeddings layer
    - Dropout
    - Dense (10) -> Output
    """
    model = Sequential([
        # Première couche convolutive
        Conv2D(32, kernel_size=(3, 3), activation='relu', input_shape=(28, 28, 1), name='conv1'),
        MaxPooling2D(pool_size=(2, 2)),
        
        # Deuxième couche convolutive
        Conv2D(64, kernel_size=(3, 3), activation='relu', name='conv2'),
        MaxPooling2D(pool_size=(2, 2)),
        
        # Troisième couche convolutive
        Conv2D(128, kernel_size=(3, 3), activation='relu', name='conv3'),
        
        # Aplatissement
        Flatten(),
        
        # Couche dense (embeddings)
        Dense(128, activation='relu', name='embeddings'),
        Dropout(0.5),
        
        # Couche de sortie
        Dense(num_classes, activation='softmax', name='output')
    ])
    
    return model

# Création du modèle
model = create_cnn_model()

# Compilation
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# Affichage du résumé
model.summary()

## 4. Entraînement du Modèle

In [ ]:
# Paramètres d'entraînement
batch_size = 128
epochs = 10

# Entraînement
history = model.fit(
    X_train, y_train_cat,
    batch_size=batch_size,
    epochs=epochs,
    validation_split=0.1,
    verbose=1
)

In [ ]:
# Visualisation de l'historique d'entraînement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(history.history['loss'], label='Train Loss')
axes[0].plot(history.history['val_loss'], label='Validation Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Evolution de la Loss')
axes[0].legend()
axes[0].grid(True)

# Accuracy
axes[1].plot(history.history['accuracy'], label='Train Accuracy')
axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Evolution de l\'Accuracy')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## 5. Évaluation du Modèle

In [ ]:
# Évaluation sur l'ensemble de test
test_loss, test_accuracy = model.evaluate(X_test, y_test_cat, verbose=0)

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")

## 6. Extraction des Embeddings

In [ ]:
# Création d'un modèle qui extrait les embeddings
embedding_model = Model(
    inputs=model.input,
    outputs=model.get_layer('embeddings').output
)

# Extraction des embeddings pour l'ensemble de test
# Utilisons seulement un sous-ensemble pour la visualisation (plus rapide)
n_samples = 5000
X_sample = X_test[:n_samples]
y_sample = y_test[:n_samples]

embeddings = embedding_model.predict(X_sample)

print(f"Forme des embeddings: {embeddings.shape}")
print(f"Dimension de l'espace d'embeddings: {embeddings.shape[1]}")

## 7. Visualisation des Embeddings avec t-SNE

In [ ]:
# Réduction de dimension avec t-SNE
print("Application de t-SNE... (cela peut prendre quelques minutes)")
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000, verbose=1)
embeddings_tsne = tsne.fit_transform(embeddings)

print(f"Forme des embeddings t-SNE: {embeddings_tsne.shape}")

In [ ]:
# Visualisation des embeddings t-SNE
plt.figure(figsize=(14, 10))

# Création d'une palette de couleurs
colors = plt.cm.tab10(np.linspace(0, 1, 10))

for digit in range(10):
    mask = y_sample == digit
    plt.scatter(
        embeddings_tsne[mask, 0],
        embeddings_tsne[mask, 1],
        c=[colors[digit]],
        label=str(digit),
        alpha=0.6,
        s=20
    )

plt.title('Visualisation des Embeddings CNN MNIST avec t-SNE', fontsize=16, fontweight='bold')
plt.xlabel('t-SNE Dimension 1', fontsize=12)
plt.ylabel('t-SNE Dimension 2', fontsize=12)
plt.legend(title='Chiffre', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('embeddings_tsne.png', dpi=300, bbox_inches='tight')
plt.show()

print("Figure sauvegardée: embeddings_tsne.png")

## 8. Visualisation des Embeddings avec UMAP (si disponible)

In [ ]:
if UMAP_AVAILABLE:
    # Réduction de dimension avec UMAP
    print("Application de UMAP...")
    reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
    embeddings_umap = reducer.fit_transform(embeddings)
    
    print(f"Forme des embeddings UMAP: {embeddings_umap.shape}")
    
    # Visualisation des embeddings UMAP
    plt.figure(figsize=(14, 10))
    
    for digit in range(10):
        mask = y_sample == digit
        plt.scatter(
            embeddings_umap[mask, 0],
            embeddings_umap[mask, 1],
            c=[colors[digit]],
            label=str(digit),
            alpha=0.6,
            s=20
        )
    
    plt.title('Visualisation des Embeddings CNN MNIST avec UMAP', fontsize=16, fontweight='bold')
    plt.xlabel('UMAP Dimension 1', fontsize=12)
    plt.ylabel('UMAP Dimension 2', fontsize=12)
    plt.legend(title='Chiffre', bbox_to_anchor=(1.05, 1), loc='upper left')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig('embeddings_umap.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Figure sauvegardée: embeddings_umap.png")
else:
    print("UMAP n'est pas disponible. Pour l'installer: pip install umap-learn")

## 9. Comparaison t-SNE vs UMAP (si UMAP disponible)

In [ ]:
if UMAP_AVAILABLE:
    fig, axes = plt.subplots(1, 2, figsize=(24, 10))
    
    # t-SNE
    for digit in range(10):
        mask = y_sample == digit
        axes[0].scatter(
            embeddings_tsne[mask, 0],
            embeddings_tsne[mask, 1],
            c=[colors[digit]],
            label=str(digit),
            alpha=0.6,
            s=20
        )
    axes[0].set_title('t-SNE', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Dimension 1')
    axes[0].set_ylabel('Dimension 2')
    axes[0].legend(title='Chiffre')
    axes[0].grid(True, alpha=0.3)
    
    # UMAP
    for digit in range(10):
        mask = y_sample == digit
        axes[1].scatter(
            embeddings_umap[mask, 0],
            embeddings_umap[mask, 1],
            c=[colors[digit]],
            label=str(digit),
            alpha=0.6,
            s=20
        )
    axes[1].set_title('UMAP', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('Dimension 1')
    axes[1].set_ylabel('Dimension 2')
    axes[1].legend(title='Chiffre')
    axes[1].grid(True, alpha=0.3)
    
    plt.suptitle('Comparaison des méthodes de réduction de dimension', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('embeddings_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    print("Figure sauvegardée: embeddings_comparison.png")

## 10. Analyse de quelques exemples

In [ ]:
# Prédictions sur l'ensemble de test
predictions = model.predict(X_sample)
predicted_classes = np.argmax(predictions, axis=1)

# Trouver des exemples mal classés
misclassified = np.where(predicted_classes != y_sample)[0]

print(f"Nombre d'exemples mal classés: {len(misclassified)} sur {len(y_sample)}")
print(f"Taux d'erreur: {len(misclassified)/len(y_sample)*100:.2f}%")

# Affichage de quelques exemples mal classés
if len(misclassified) > 0:
    n_display = min(10, len(misclassified))
    fig, axes = plt.subplots(2, 5, figsize=(15, 6))
    
    for i, ax in enumerate(axes.flat):
        if i < n_display:
            idx = misclassified[i]
            ax.imshow(X_sample[idx].reshape(28, 28), cmap='gray')
            ax.set_title(f'Vrai: {y_sample[idx]}, Prédit: {predicted_classes[idx]}')
            ax.axis('off')
    
    plt.suptitle('Exemples mal classés', fontsize=16)
    plt.tight_layout()
    plt.show()

## Conclusions

### Ce que nous avons accompli :

1. **Construction d'un CNN** : Nous avons créé un réseau de neurones convolutif avec 3 couches convolutives et une couche dense d'embeddings de dimension 128.

2. **Entraînement** : Le modèle a été entraîné sur 60,000 images MNIST avec une bonne performance.

3. **Extraction des Embeddings** : Nous avons extrait les représentations apprises par le réseau à partir de la dernière couche cachée.

4. **Visualisation** : 
   - **t-SNE** : Montre comment les différents chiffres se regroupent dans l'espace des embeddings
   - **UMAP** : Alternative plus rapide qui préserve mieux la structure globale

### Observations :

- Les clusters de chiffres similaires (comme 4 et 9, ou 3 et 8) sont souvent proches dans l'espace des embeddings
- Le CNN apprend des représentations qui séparent bien les différentes classes
- La visualisation aide à comprendre les erreurs de classification

### Pour aller plus loin :

1. Essayer différentes architectures CNN
2. Utiliser le data augmentation
3. Tester d'autres dimensions d'embeddings
4. Appliquer ces techniques à d'autres datasets (Fashion-MNIST, CIFAR-10, etc.)